# Conda-PyPI Failures Analysis

In [1]:
import sqlite3
from pathlib import Path


# Kernel cwd varies by how the notebook was opened (repo root vs. notebooks/);
# resolve the repo root by walking up until we find data/v.db, instead of
# assuming Path.cwd() already is it (matches parselmouth_algo.ipynb).
def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "data" / "v.db").exists():
            return candidate
    raise FileNotFoundError(f"could not find data/v.db above {start}")


ROOT = _find_repo_root(Path.cwd())
V_DB = ROOT / "data" / "v.db"  # this repo's PyPI corpus; see the Makefile

# Genuinely read-only: `reroll_data.db.connect(read_only=True)` only skips a
# mkdir, it still opens for writing. A crawl/backfill may be running against
# this file, so a diagnostic notebook must not be able to touch it -- same
# `mode=ro` URI connection as `reroll_data.investigate.connect_ro`.
if not V_DB.is_file():
    raise SystemExit(f"corpus not found: {V_DB}")

v = sqlite3.connect(f"file:{V_DB}?mode=ro", uri=True)
v.execute("PRAGMA busy_timeout=60000")

print("repo root:", ROOT)
print("v.db:", V_DB, f"({V_DB.stat().st_size / 1e9:.2f} GB)")

repo root: /Users/anil/code/reroll-data
v.db: /Users/anil/code/reroll-data/data/v.db (50.61 GB)


## conda-pypi failures

`repodata_conversion.conda_pypi_error` is set by `repodata_convert._convert_one`:
every failure -- `WheelNotFound`, `MetadataUnavailable`, `NotPureWheel`, or
anything conda-pypi itself raises while parsing a malformed `METADATA` body --
is caught and stored as `f"{type(exc).__name__}: {exc}"[:1000]` (see
`reroll_data.repodata_convert`). So each row's error already carries a clean
exception-type prefix to split on; classification below buckets first by that
type, then by message shape within the biggest ones.

In [4]:
### 1. Overview

n_compatible = v.execute(
    "SELECT count(*) FROM repodata_conversion WHERE conda_pypi_compatible = 1"
).fetchone()[0]
n_attempted = v.execute(
    "SELECT count(*) FROM repodata_conversion "
    "WHERE conda_pypi_data IS NOT NULL OR conda_pypi_error IS NOT NULL"
).fetchone()[0]
n_ok = v.execute(
    "SELECT count(*) FROM repodata_conversion WHERE conda_pypi_data IS NOT NULL"
).fetchone()[0]
n_error = v.execute(
    "SELECT count(*) FROM repodata_conversion WHERE conda_pypi_error IS NOT NULL"
).fetchone()[0]

print(f"conda-pypi-compatible wheels:  {n_compatible:>12,}")
print(
    f"attempted so far:              {n_attempted:>12,}  ({n_attempted / n_compatible:.1%} of compatible)"
)
print(
    f"  succeeded:                   {n_ok:>12,}  ({n_ok / n_attempted:.1%} of attempted)"
)
print(
    f"  errored:                     {n_error:>12,}  ({n_error / n_attempted:.1%} of attempted)"
)

conda-pypi-compatible wheels:     6,382,022
attempted so far:                 6,382,022  (100.0% of compatible)
  succeeded:                      6,371,622  (99.8% of attempted)
  errored:                           10,400  (0.2% of attempted)


In [8]:
### 2. Load errors into a dataframe, split into (exc_type, message)

import re

import polars as pl

_TYPE_RE = re.compile(r"^([A-Za-z_][A-Za-z0-9_.]*): (.*)$", re.DOTALL)


def split_error(e: str) -> tuple[str, str]:
    m = _TYPE_RE.match(e)
    return m.groups() if m else ("UNPARSEABLE", e)


rows = v.execute(
    "SELECT project, filename, conda_pypi_error FROM repodata_conversion "
    "WHERE conda_pypi_error IS NOT NULL"
).fetchall()

errors = pl.DataFrame(rows, schema=["project", "filename", "error"], orient="row")
errors = errors.with_columns(
    pl.col("error")
    .map_elements(lambda e: split_error(e)[0], return_dtype=pl.String)
    .alias("exc_type"),
    pl.col("error")
    .map_elements(lambda e: split_error(e)[1], return_dtype=pl.String)
    .alias("message"),
)
print(f"{errors.height:,} error rows loaded")

exc_type_counts = (
    errors.group_by("exc_type")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .with_columns((pl.col("count") / errors.height).alias("share"))
)
display(exc_type_counts)

10,400 error rows loaded


exc_type,count,share
str,u32,f64
"""InvalidRequirement""",10248,0.985385
"""MetadataUnavailable""",151,0.014519
"""UnicodeDecodeError""",1,0.000096


In [9]:
### 3. Sample messages per exc_type, to design sub-bucket rules

with pl.Config(fmt_str_lengths=160, tbl_rows=8):
    for exc_type in exc_type_counts["exc_type"]:
        print(f"=== {exc_type} ===")
        display(errors.filter(pl.col("exc_type") == exc_type).select("message").head(4))

=== InvalidRequirement ===


=== MetadataUnavailable ===


message
str
"""Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier python-slugify (==4.0.1requests==2.24.0) ~~~~~~~~~^"""
"""Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier tensorflow-addons (>=0.11.0keras-tcn) ~~~~~~~~~^"""
""".* suffix can only be used with `==` or `!=` operators matplotlib>=3.6.* ~~~~~~^"""
""".* suffix can only be used with `==` or `!=` operators matplotlib>=3.6.* ~~~~~~^"""


=== MetadataUnavailable ===


=== UnicodeDecodeError ===


message
str
"""wheel has no PEP 658 metadata sidecar recorded (metadata_sha256 IS NULL)"""
"""wheel has no PEP 658 metadata sidecar recorded (metadata_sha256 IS NULL)"""
"""wheel has no PEP 658 metadata sidecar recorded (metadata_sha256 IS NULL)"""
"""wheel has no PEP 658 metadata sidecar recorded (metadata_sha256 IS NULL)"""


=== UnicodeDecodeError ===


message
str
"""'utf-8' codec can't decode byte 0x82 in position 1851: invalid start byte"""


In [10]:
### 4. Sub-bucket InvalidRequirement (98.5% of errors) by the packaging parser's reason line

# The message is packaging.requirements.InvalidRequirement's own rendering:
# reason line, then the offending requirement string, then a "~~~^" caret
# line pointing at the failure column. Only the reason line is shared across
# occurrences (the requirement string/caret vary per wheel), so classify on
# that first line.
_INVALID_REQ_RULES = [
    (
        "mismatched_parenthesis",
        r"^Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS",
    ),
    ("wildcard_suffix_misuse", r"^\.\* suffix can only be used"),
    ("expected_package_name", r"^Expected package name at the start"),
    ("expected_marker_or_string", r"^Expected a marker variable or quoted string"),
    ("expected_end_or_semicolon", r"^Expected end or semicolon"),
    ("invalid_specifier", r"^Invalid specifier:"),
    ("local_version_label_misuse", r"^Local version label can only be used"),
    ("expected_comma_or_end", r"^Expected comma or end"),
    ("expected_string", r"^Expected string"),
    ("expected_version", r"^Expected version"),
]


def classify_invalid_requirement(msg: str) -> str:
    reason_line = msg.split("\n", 1)[0].strip()
    for sub_type, pattern in _INVALID_REQ_RULES:
        if re.match(pattern, reason_line):
            return sub_type
    return "other"


# `bucket`: for InvalidRequirement, the parser sub-reason; for every other
# exc_type, the exc_type itself (they're small and already homogeneous --
# see §3).
errors = errors.with_columns(
    pl.when(pl.col("exc_type") == "InvalidRequirement")
    .then(
        pl.col("message").map_elements(
            classify_invalid_requirement, return_dtype=pl.String
        )
    )
    .otherwise(pl.col("exc_type"))
    .alias("bucket")
)

buckets = (
    errors.group_by("exc_type", "bucket")
    .agg(pl.len().alias("count"), pl.col("message").first().alias("example"))
    .sort("count", descending=True)
    .with_columns((pl.col("count") / errors.height).alias("share"))
)

with pl.Config(fmt_str_lengths=90, tbl_rows=20):
    display(buckets.select("exc_type", "bucket", "count", "share", "example"))

exc_type,bucket,count,share,example
str,str,u32,f64,str
"""InvalidRequirement""","""mismatched_parenthesis""",7366,0.708269,"""Expected matching RIGHT_PARENTHESIS for LEFT_PARENTHESIS, after version specifier pyth…"
"""InvalidRequirement""","""wildcard_suffix_misuse""",2663,0.256058,""".* suffix can only be used with `==` or `!=` operators matplotlib>=3.6.* …"
"""InvalidRequirement""","""local_version_label_misuse""",197,0.018942,"""Local version label can only be used with `==` or `!=` operators torch (>=1.13.0+cu117…"
"""MetadataUnavailable""","""MetadataUnavailable""",151,0.014519,"""wheel has no PEP 658 metadata sidecar recorded (metadata_sha256 IS NULL)"""
"""InvalidRequirement""","""expected_end_or_semicolon""",21,0.002019,"""Expected end or semicolon (after name and no valid version specifier) pydantic<.2.0.0,…"
"""InvalidRequirement""","""expected_marker_or_string""",1,0.000096,"""Expected a marker variable or quoted string enum34; python_version < 3.4 …"
"""UnicodeDecodeError""","""UnicodeDecodeError""",1,0.000096,"""'utf-8' codec can't decode byte 0x82 in position 1851: invalid start byte"""


In [11]:
### 5. Summary: all conda-pypi failure buckets

summary = buckets.with_columns(
    (pl.col("share") * 100).round(1).alias("share_pct")
).select("bucket", "exc_type", "count", "share_pct")

print(
    f"{len(errors):,} conda-pypi failures across {n_compatible:,} compatible wheels ({n_error / n_compatible:.2%})\n"
)
print(f"{'bucket':<28} {'exc_type':<20} {'count':>7}  {'share':>6}")
print("-" * 66)
for row in summary.iter_rows(named=True):
    print(
        f"{row['bucket']:<28} {row['exc_type']:<20} {row['count']:>7,}  {row['share_pct']:>5.1f}%"
    )

10,400 conda-pypi failures across 6,382,022 compatible wheels (0.16%)

bucket                       exc_type               count   share
------------------------------------------------------------------
mismatched_parenthesis       InvalidRequirement     7,366   70.8%
wildcard_suffix_misuse       InvalidRequirement     2,663   25.6%
local_version_label_misuse   InvalidRequirement       197    1.9%
MetadataUnavailable          MetadataUnavailable      151    1.5%
expected_end_or_semicolon    InvalidRequirement        21    0.2%
expected_marker_or_string    InvalidRequirement         1    0.0%
UnicodeDecodeError           UnicodeDecodeError         1    0.0%
